# Analisi probabilistica e Bayes con Python su Data Warehouse TechStore
## Ripasso Teorico
- Probabilità
- Probabilità condizionata
- Tabelle di contingenza
- Teorema di Bayes


In [ ]:
import psycopg, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
conn = psycopg.connect(host='its_postgresql', port=5432, dbname='techstore_dw', user='postgres', password='postgres')
print('Connessione OK')

## Esercizio 1 – Distribuzione delle categorie prodotto

In [ ]:
df = pd.read_sql('''SELECT categoria, COUNT(*) n FROM fact_vendite fv JOIN dim_prodotto dp ON fv.prodotto_sk=dp.prodotto_sk GROUP BY categoria''', conn)
df['prob'] = df['n']/df['n'].sum()
df

## Esercizio 2 – Distribuzione della fascia di età

In [ ]:
df_eta = pd.read_sql('SELECT fascia_eta, COUNT(*) n FROM dim_cliente dc JOIN fact_vendite fv ON dc.cliente_sk=fv.cliente_sk GROUP BY fascia_eta', conn)
df_eta['prob']=df_eta['n']/df_eta['n'].sum()
df_eta

## Esercizio 3 – Tabella di contingenza Categoria × Fascia Età

In [ ]:
df_ct = pd.read_sql('''SELECT fascia_eta, categoria, COUNT(*) n FROM fact_vendite fv JOIN dim_cliente dc ON fv.cliente_sk=dc.cliente_sk JOIN dim_prodotto dp ON fv.prodotto_sk=dp.prodotto_sk GROUP BY fascia_eta, categoria''', conn)
pivot=df_ct.pivot(index='fascia_eta', columns='categoria', values='n')
sns.heatmap(pivot, annot=True, fmt='d')

## Esercizio 4 – Probabilità condizionata: P(Laptop | 18–25)

In [ ]:
# P(Laptop | 18–25)
df_ct
# (completare con P(A|B)=P(A,B)/P(B))
# Esercizio 4 – Probabilità condizionata: P(Categoria | Fascia Età)

# Seleziono la riga corrispondente alla fascia di età desiderata
# In questo esempio consideriamo la fascia 18–25
riga = pivot.loc['18-24']

# Calcolo il totale delle vendite per quella fascia
totale_fascia = riga.sum()

# Probabilità condizionata: vendite per categoria / totale fascia
prob_condizionata = riga / totale_fascia

# Mostro il risultato
prob_condizionata

p_laptop = prob_condizionata['Laptop']
print(f"P(Laptop | 18-24) = {p_laptop:.4f}")




## Esercizio 5 – Probabilità condizionata inversa: P(18–25 | Smartphone)

In [ ]:
# P(18–25 | Smartphone)
# usare tabella df_ct
df_ct

#seleziono la colonna corrispondente a Smartphone
colonna = pivot['Smartphone']

totale_fascia = colonna.sum()

prob_condizionata = colonna / totale_fascia

prob_condizionata['18-24']

## Esercizio 6 – doppia condizione: P(Smartphone | 18–25 AND fascia eta)

In [ ]:
df = pd.read_sql('''
    SELECT 
        dc.fascia_eta,
        dp.categoria,
        sum (fv.ricavi)/ sum (fv.quantita) AS costo_medio_categoria,
        COUNT(*) AS n
    FROM fact_vendite fv
    JOIN dim_cliente dc ON fv.cliente_sk = dc.cliente_sk
    JOIN dim_prodotto dp ON fv.prodotto_sk = dp.prodotto_sk
    GROUP BY dc.fascia_eta, dp.categoria;
''', conn)

# Fasce di costo prodotto per categoria
df['fascia_costo'] = pd.cut(
    df['costo_medio_categoria'],
    bins=[0, 300, 800, 2000],
    labels=['basso', 'medio', 'alto']
)

# Esempio: P(Laptop | fascia 18–24 AND fascia costo “alto”)
fascia = "18-24"
fascia_costo = "alto"
categoria = "Laptop"

subset = df[
    (df['fascia_eta']==fascia) &
    (df['fascia_costo']==fascia_costo)
]

totale = subset['n'].sum()
n_categoria = subset.loc[subset['categoria']==categoria, 'n'].sum()

p = n_categoria / totale
p


## Esercizio 7 – Bayes: P(Smartphone | 18–25)

In [ ]:
# ============================
# ESERCIZIO 7 – BAYES
# Calcolare P(Smartphone | 18–24)
# ============================

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1) Estraggo dati categoria × fascia età
df_ct = pd.read_sql('''
    SELECT 
        dc.fascia_eta,
        dp.categoria,
        COUNT(*) AS n
    FROM fact_vendite fv
    JOIN dim_cliente dc ON fv.cliente_sk = dc.cliente_sk
    JOIN dim_prodotto dp ON fv.prodotto_sk = dp.prodotto_sk
    GROUP BY dc.fascia_eta, dp.categoria;
''', conn)

# Costruisco la tabella pivot (tabella di contingenza)
pivot = df_ct.pivot(index='fascia_eta', columns='categoria', values='n').fillna(0)

In [ ]:
# ======================================================
# STEP 1 — PRIOR: P(Smartphone)
# ======================================================

df_cat = df_ct.groupby('categoria')['n'].sum().reset_index()
df_cat['prob'] = df_cat['n'] / df_cat['n'].sum()

p_prior = df_cat.loc[df_cat['categoria']=="Smartphone", 'prob'].iloc[0]
print("Prior P(Smartphone) =", p_prior)

In [ ]:
# ======================================================
# STEP 2 — LIKELIHOOD: P(18–24 | Smartphone)
# ======================================================

col = pivot['Smartphone']   # colonna della categoria Smartphone
p_likelihood = col.loc['18-24'] / col.sum()

print("Likelihood P(18–24 | Smartphone) =", p_likelihood)

In [ ]:
# ======================================================
# STEP 3 — EVIDENCE: P(18–24)
# ======================================================

df_eta = df_ct.groupby('fascia_eta')['n'].sum().reset_index()
df_eta['prob'] = df_eta['n'] / df_eta['n'].sum()

p_evidence = df_eta.loc[df_eta['fascia_eta']=="18-24", 'prob'].iloc[0]
print("Evidence P(18–24) =", p_evidence)

In [ ]:
# ======================================================
# STEP 4 — POSTERIOR: Bayes
# ======================================================

p_posterior = (p_likelihood * p_prior) / p_evidence
print("\nPosterior P(Smartphone | 18–24) =", p_posterior)